## 1. Configuration and hardware checks

The default smoke configuration is suitable for inspecting the notebook on a CPU-only machine. Set `RUN_TRAINING = True` on a machine with an appropriate CUDA GPU or TPU, after confirming Gemma access and available storage.

In [ ]:
from pathlib import Path
import json
import os
import random
import time

SEED = 13104
RUN_TRAINING = False
USE_SAMPLE_DATA = True
SAMPLE_TRAIN = 128
SAMPLE_VALIDATION = 64
SAMPLE_TEST = 64
MAX_LENGTH = 256
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
RANKS_TO_TEST = [2, 4, 8]
BASE_MODEL = "gemma_7b_en"
HF_MODEL = os.getenv("HF_MODEL", "google/gemma-2b")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print({
    "python": __import__("sys").version.split()[0],
    "run_training": RUN_TRAINING,
    "base_model": BASE_MODEL,
    "hf_model": HF_MODEL,
})

In [ ]:
import importlib.util
import platform

def package_available(name):
    return importlib.util.find_spec(name) is not None

capabilities = {
    "tensorflow": package_available("tensorflow"),
    "keras_hub": package_available("keras_hub"),
    "torch": package_available("torch"),
    "transformers": package_available("transformers"),
    "peft": package_available("peft"),
    "bitsandbytes": package_available("bitsandbytes"),
    "datasets": package_available("datasets"),
}
print("Platform:", platform.platform())
print("Available packages:", capabilities)

if capabilities["torch"]:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
else:
    print("PyTorch is not installed; QLoRA cells will remain unavailable.")

In [ ]:
def configure_tpu_strategy():
    """Create a TPU distribution strategy when a TPU runtime is available."""
    if not capabilities["tensorflow"]:
        return None
    import tensorflow as tf
    try:
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)
        return tf.distribute.TPUStrategy(resolver)
    except ValueError:
        return None


tpu_strategy = configure_tpu_strategy()
print("TPU strategy active:", tpu_strategy is not None)

## 2. Load and prepare the IMDB Reviews dataset

The dataset is split into training, validation, and test portions. The sample limits are only for smoke testing; remove them for the full experiment.

In [ ]:
if not package_available("datasets"):
    raise ImportError("Install the `datasets` package before loading IMDB.")

from datasets import load_dataset

imdb = load_dataset("imdb")
if USE_SAMPLE_DATA:
    train = imdb["train"].shuffle(seed=SEED).select(range(min(SAMPLE_TRAIN, len(imdb["train"]))))
    validation = imdb["test"].shuffle(seed=SEED).select(range(min(SAMPLE_VALIDATION, len(imdb["test"]))))
    test = imdb["test"].shuffle(seed=SEED + 1).select(range(min(SAMPLE_TEST, len(imdb["test"]))))
else:
    split = imdb["train"].train_test_split(test_size=0.1, seed=SEED)
    train, validation, test = split["train"], split["test"], imdb["test"]

print(len(train), len(validation), len(test))
print(train[0])

## 3. Tokenization and dynamic batching

The Keras path uses a KerasHub tokenizer. The PEFT path uses a Transformers tokenizer and `DataCollatorWithPadding`, which pads each batch to its longest sequence rather than padding the entire dataset to one global length.

In [ ]:
def make_hf_features(batch, tokenizer):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False)

hf_tokenizer = None
tokenized_train = tokenized_validation = tokenized_test = None
if package_available("transformers"):
    from transformers import AutoTokenizer, DataCollatorWithPadding
    hf_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
    if hf_tokenizer.pad_token is None:
        hf_tokenizer.pad_token = hf_tokenizer.eos_token
    tokenized_train = train.map(lambda batch: make_hf_features(batch, hf_tokenizer), batched=True)
    tokenized_validation = validation.map(lambda batch: make_hf_features(batch, hf_tokenizer), batched=True)
    tokenized_test = test.map(lambda batch: make_hf_features(batch, hf_tokenizer), batched=True)
    collator = DataCollatorWithPadding(tokenizer=hf_tokenizer, return_tensors="pt")
    print(collator([tokenized_train[0], tokenized_train[1]])["input_ids"].shape)
else:
    print("Transformers is unavailable; tokenizer and collator setup is deferred.")

## 4. Load Gemma with Keras/KerasHub and configure LoRA

`gemma_7b_en` is the preset named by the assignment. KerasHub may require a Kaggle credential or an accepted model license. The base model is frozen before enabling LoRA adapters.

In [ ]:
keras_model = None
if package_available("keras_hub") and package_available("keras"):
    import keras
    import keras_hub
    try:
        keras_model = keras_hub.models.GemmaCausalLM.from_preset(BASE_MODEL)
        print("Loaded", BASE_MODEL)
    except Exception as error:
        print("Gemma could not be loaded in this environment:", error)
else:
    print("Install keras and keras-hub to load the Keras Gemma preset.")

def configure_keras_lora(model, rank=LORA_RANK, alpha=LORA_ALPHA):
    if model is None:
        return None
    if hasattr(model, "backbone") and hasattr(model.backbone, "enable_lora"):
        model.backbone.enable_lora(rank=rank)
    elif hasattr(model, "enable_lora"):
        model.enable_lora(rank=rank)
    else:
        raise AttributeError("This KerasHub version does not expose a LoRA enable method.")
    for layer in model.layers:
        if hasattr(layer, "trainable"):
            layer.trainable = False
    return model

if RUN_TRAINING and keras_model is not None:
    keras_model = configure_keras_lora(keras_model)
    keras_model.summary()

## 5. LoRA training loop, checkpoints, and validation

This cell shows the Keras training configuration. The exact prompt format can be adjusted for the selected Gemma preset. It is intentionally guarded because model construction and training can require many gigabytes of accelerator memory.

In [ ]:
if RUN_TRAINING and keras_model is not None:
    import keras
    keras_model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=2e-5, weight_decay=0.01),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(ARTIFACT_DIR / "keras_lora_epoch_{epoch:02d}.weights.h5",),
            save_weights_only=True,
        ),
        keras.callbacks.ReduceLROnPlateau(patience=1),
    ]
    # Convert tokenized examples to the input/label tensors expected by the preset.
    # The exact packing depends on the KerasHub version and is documented in README.md.
    print("Keras LoRA model is configured; prepare packed causal-LM tensors before calling fit().")
else:
    print("LoRA training skipped. Set RUN_TRAINING=True on a supported accelerator.")

## 6. Baseline and LoRA evaluation

Accuracy and F1 are computed from sentiment predictions on the same validation examples. Results stay empty until inference is actually run.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

def metric_row(name, y_true, y_pred, trainable_parameters=None, peak_memory_mb=None, training_seconds=None):
    return {
        "model": name,
        "trainable_parameters": trainable_parameters,
        "peak_gpu_memory_mb": peak_memory_mb,
        "training_seconds": training_seconds,
        "validation_accuracy": accuracy_score(y_true, y_pred),
        "validation_f1": f1_score(y_true, y_pred),
    }

evaluation_rows = []
# Example after real inference: evaluation_rows.append(metric_row("base", labels, predictions))
comparison = pd.DataFrame(evaluation_rows, columns=[
    "model", "trainable_parameters", "peak_gpu_memory_mb",
    "training_seconds", "validation_accuracy", "validation_f1"
])
comparison

## 7. Save and reload LoRA adapter weights

Adapters must be saved separately from the frozen base model. The helper checks the installed KerasHub API so an API mismatch is visible instead of silently producing a misleading artifact.

In [ ]:
def save_keras_lora_adapter(model, path):
    if model is None:
        raise ValueError("No Keras model is loaded.")
    saver = getattr(model, "save_lora_weights", None)
    if saver is None:
        raise AttributeError("Use the save_lora_weights API provided by your KerasHub version.")
    saver(path)

def load_keras_lora_adapter(model, path):
    loader = getattr(model, "load_lora_weights", None)
    if loader is None:
        raise AttributeError("Use the load_lora_weights API provided by your KerasHub version.")
    loader(path)

if not RUN_TRAINING:
    print("Adapter save/reload helper defined; skipped until a trained adapter exists.")

## 8. QLoRA with PEFT and BitsAndBytes

QLoRA uses NF4 4-bit quantization, double quantization, float16 computation, and rank 8 adapters. BitsAndBytes requires a compatible CUDA environment; it is not a CPU fallback.

In [ ]:
def build_qlora_model(rank=LORA_RANK):
    required = ["torch", "transformers", "peft", "bitsandbytes"]
    missing = [name for name in required if not package_available(name)]
    if missing:
        raise ImportError("Missing QLoRA packages: " + ", ".join(missing))
    import torch
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, TaskType, get_peft_model
    if not torch.cuda.is_available():
        raise RuntimeError("QLoRA with BitsAndBytes requires a CUDA GPU.")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    base = AutoModelForCausalLM.from_pretrained(
        HF_MODEL,
        quantization_config=quantization_config,
        device_map="auto",
    )
    lora_config = LoraConfig(
        r=rank,
        lora_alpha=2 * rank,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type=TaskType.CAUSAL_LM,
    )
    return get_peft_model(base, lora_config)

qlora_model = None
if RUN_TRAINING:
    qlora_model = build_qlora_model(rank=LORA_RANK)
    qlora_model.print_trainable_parameters()
else:
    print("QLoRA construction skipped. Set RUN_TRAINING=True on a CUDA machine.")

## 9. QLoRA training, memory, and evaluation

The Trainer uses the same tokenized IMDB splits and dynamic collator. Peak memory and elapsed time are captured only around a real training run.

In [ ]:
def train_qlora(model, train_dataset, validation_dataset):
    import torch
    from transformers import TrainingArguments, Trainer
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    args = TrainingArguments(
        output_dir=str(ARTIFACT_DIR / "qlora"),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        num_train_epochs=1,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        fp16=True,
        report_to="none",
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=train_dataset,
        eval_dataset=validation_dataset, data_collator=collator,
    )
    trainer.train()
    elapsed = time.perf_counter() - started
    peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 2)
    model.save_pretrained(ARTIFACT_DIR / "qlora_adapter")
    return trainer, elapsed, peak_memory

if RUN_TRAINING and qlora_model is not None:
    qlora_trainer, qlora_seconds, qlora_peak_memory = train_qlora(
        qlora_model, tokenized_train, tokenized_validation
    )
else:
    print("QLoRA training skipped; no measurements are reported.")

## 10. LoRA rank sweep and comparison plot

Run each rank with the same data split and training budget. The table remains empty until the sweep is executed, preventing placeholder numbers from being mistaken for results.

In [ ]:
rank_results = []

def append_rank_result(method, rank, trainable_parameters, validation_f1):
    rank_results.append({
        "method": method,
        "rank": rank,
        "trainable_parameters": trainable_parameters,
        "validation_f1": validation_f1,
    })

if RUN_TRAINING:
    for rank in RANKS_TO_TEST:
        # Rebuild, train, evaluate, and call append_rank_result for each method.
        print("Run rank", rank, "for both LoRA and QLoRA with identical settings.")
else:
    print("Rank sweep skipped; enable training to collect measured values.")

rank_frame = pd.DataFrame(rank_results)
if not rank_frame.empty:
    import matplotlib.pyplot as plt
    for method, group in rank_frame.groupby("method"):
        plt.plot(group["trainable_parameters"], group["validation_f1"], marker="o", label=method)
    plt.xlabel("Trainable parameters")
    plt.ylabel("Validation F1-score")
    plt.title("LoRA rank trade-off")
    plt.legend()
    plt.tight_layout()
    plt.savefig(ARTIFACT_DIR / "rank_tradeoff.png", dpi=160)
    plt.show()
rank_frame

## 11. Qualitative generation comparison

Use one identical prompt with the pretrained base model, the LoRA adapter, and the QLoRA adapter. Save the generated text only after each model has actually been loaded.

In [ ]:
EVALUATION_PROMPT = (
    "Review: The acting was convincing and the story kept me interested."
)

def generate_comparison(prompt, models):
    outputs = []
    for name, generator in models.items():
        outputs.append({"model": name, "prompt": prompt, "output": generator(prompt)})
    return pd.DataFrame(outputs)

generation_results = pd.DataFrame(columns=["model", "prompt", "output"])
print("Generation prompt:", EVALUATION_PROMPT)
print("Load the base, LoRA, and QLoRA generators here after their adapters are available.")

## Conclusion and limitations

LoRA trains a small set of adapter parameters while keeping the base model frozen. QLoRA additionally stores the base weights in 4-bit NF4 form, reducing memory at the cost of quantization overhead and a CUDA/BitsAndBytes dependency.

This notebook provides the complete experiment workflow and leaves measured tables empty until execution on suitable hardware. The README documents the expected environment and the exact commands for running the notebook.